# Phase 12 — Blur Robustness Training
**RealCentric Generator-Agnostic Deepfake Detection**

Addresses the regression in MLP performance under heavy blur (k=7) after switching to multi-dataset training.
This extracts explicitly blurred features from the CelebDF training split and retrains the model.

In [ ]:
import sys, cv2, numpy as np, time, random
import matplotlib.pyplot as plt
sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from config.config_loader import load_config
from src.features.extractor import FeatureFusionPipeline
import shutil

BASE     = Path('/data/mpstme-naman/deepfake_detection')
PROC     = BASE / 'data' / 'processed'
FEAT_DIR = BASE / 'data' / 'features'
CKPT_DIR = BASE / 'checkpoints'
RES_DIR  = BASE / 'results'

FAST_RUN = True  # Toggle for fast prototyping (10% CelebDF)

out_feat = FEAT_DIR / ('Z_train_multi_blur_fast.npy' if FAST_RUN else 'Z_train_multi_blur.npy')
out_lbl  = FEAT_DIR / ('y_train_multi_blur_fast.npy' if FAST_RUN else 'y_train_multi_blur.npy')


## Step 1 — Image Slicing & Feature Loading

In [ ]:
if out_feat.exists() and out_lbl.exists():
    print(f"Augmented features already exist at {out_feat.name}, skipping extraction.")
    Z_train_aug = np.load(out_feat)
    y_train_aug = np.load(out_lbl)
else:
    print("Dynamically loading and splitting CelebDF raw images to avoid train/test leakage...")
    cr = sorted((PROC/'celebdf'/'real').glob('*.png'))
    cf = sorted((PROC/'celebdf'/'fake').glob('*.png'))
    all_paths = cr + cf
    lbls = [0]*len(cr) + [1]*len(cf)
    
    # Standard 70/15/15 split using random_state=42 ensures exact match with Z_train
    paths_tr, _, y_tr_cd, _ = train_test_split(all_paths, lbls, test_size=0.30, random_state=42, stratify=lbls)
    
    frac = 0.10 if FAST_RUN else 0.30
    n_sample = int(len(paths_tr) * frac)
    
    np.random.seed(42)
    sample_indices = np.random.choice(len(paths_tr), n_sample, replace=False)
    sample_paths = [paths_tr[i] for i in sample_indices]
    sample_lbls  = [y_tr_cd[i]  for i in sample_indices]
    
    print(f"\nApplying Gaussian blur (k=3,5,7) to {n_sample:,} images...")
    aug_imgs = []
    for p in tqdm(sample_paths, desc="Blurring"):
        img = cv2.imread(str(p))
        if img is not None:
            k = int(np.random.choice([3, 5, 7]))
            img_b = cv2.GaussianBlur(img, (k, k), 0)
            aug_imgs.append(img_b)
            
    print("\nExtracting blended features through unified pipeline...")
    cfg = load_config()
    pipeline = FeatureFusionPipeline(cfg=cfg, backbone=cfg['features']['cnn']['backbones'][0])
    pipeline.set_state(np.load(CKPT_DIR / 'pipeline_state.pkl', allow_pickle=True))
    
    t0 = time.time()
    Z_blur = pipeline.extract_batch(aug_imgs, normalise=True, cnn_batch_size=64)
    print(f"Extracted in {time.time()-t0:.1f}s. Shape: {Z_blur.shape}")
    
    Z_multi = np.load(FEAT_DIR / 'Z_train_multi.npy')
    y_multi = np.load(FEAT_DIR / 'y_train_multi.npy')
    
    Z_train_aug = np.vstack([Z_multi, Z_blur])
    y_train_aug = np.hstack([y_multi, sample_lbls])
    
    np.save(out_feat, Z_train_aug)
    np.save(out_lbl, y_train_aug)
    print(f"\nSaved augmented training matrices: {Z_train_aug.shape}")


## Step 2 — Retrain MultiDatasetTrainer

In [ ]:
from src.models.mlp_classifier import MultiDatasetTrainer
cfg = load_config()
trainer = MultiDatasetTrainer(cfg=cfg, input_dim=981, pos_weight=0.34, use_dann=False)

Z_val_multi = np.load(FEAT_DIR / 'Z_val_multi.npy')
y_val_multi = np.load(FEAT_DIR / 'y_val_multi.npy')

print("\nRetraining MLP on Blur-Augmented Dataset...")
t0 = time.time()
best_auc = trainer.train(Z_train_aug, y_train_aug, Z_val_multi, y_val_multi,
                         checkpoint_dir=str(CKPT_DIR), run_name='mlp_multi_blur')
print(f"✓ Training completed in {time.time()-t0:.1f}s. Best Val AUC: {best_auc:.4f}")

## Step 3 — Evaluate Regressions & Auto-Promote Model
Promotion gated on: Extracted Blur=7 AUC $\geq 0.95$ AND Base FF++ AUC $\geq 0.98$.

In [ ]:
# 1. Base FF++
Z_ff = np.load(FEAT_DIR / 'Z_ff.npy')
y_ff = np.load(FEAT_DIR / 'y_ff.npy')
res_ff = trainer.evaluate(Z_ff, y_ff)
base_ff_auc = res_ff['auc']

# 2. Heavy Blur Simulation (Subset of FF++ due to high extraction overhead)
print("\n=== Robustness Verification (k=7) ===")
ff_paths_real = sorted((PROC/'ff'/'real').glob('*.png'))[:1000]
ff_paths_fake = sorted((PROC/'ff'/'fake').glob('*.png'))[:1000]
paths_test = ff_paths_real + ff_paths_fake
lbls_test = [0]*len(ff_paths_real) + [1]*len(ff_paths_fake)

print("Applying extreme Gaussian Blur (k=7) and extracting...")
pipeline = FeatureFusionPipeline(cfg=cfg, backbone=cfg['features']['cnn']['backbones'][0])
pipeline.set_state(np.load(CKPT_DIR / 'pipeline_state.pkl', allow_pickle=True))

b7_imgs = [cv2.GaussianBlur(cv2.imread(str(p)), (7,7), 0) for p in paths_test]
Z_test_b7 = pipeline.extract_batch(b7_imgs, normalise=True, cnn_batch_size=64)

res_b7 = trainer.evaluate(Z_test_b7, np.array(lbls_test))
blur_7_auc = res_b7['auc']

print("\n--- Gated Promotion Check ---")
print(f"FF++ Base AUC  : {base_ff_auc:.4f}  (Target >= 0.98)")
print(f"FF++ Blur=7 AUC: {blur_7_auc:.4f}  (Target >= 0.95)")

if base_ff_auc >= 0.98 and blur_7_auc >= 0.95:
    shutil.copy(CKPT_DIR / 'mlp_multi_blur_best.pt', CKPT_DIR / 'mlp_multi_best.pt')
    print("\n✅ SUCCESS: Thresholds met. Checkpoint auto-promoted to mlp_multi_best.pt!")
else:
    print("\n❌ WARNING: Thresholds not met. Auto-promotion skipped so original logic remains intact.")